In [1]:
import time
from functools import partial
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import pickle
import jax.numpy as jnp
import jax
import jax.random as jr
import numpy as onp
import jax.flatten_util
import pandas as pd
import optax
import jaxopt
from flax.training import train_state
import flax.linen as nn
import scipy
from jaxid.common import MLP
from neuralss import ss_init, ss_apply
from typing import Callable, List
# from ae import Encoder, Projector
import nonlinear_benchmarks

In [2]:
key = jr.key(42)

In [3]:
jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [4]:
class Encoder(nn.Module):
    mlp_layers: List[int]
    rnn_size: int = 128
    
    def setup(self):
        self.rnn = nn.Bidirectional(
            nn.RNN(nn.GRUCell(self.rnn_size)), nn.RNN(nn.GRUCell(self.rnn_size))
        )
        self.mlp_mean = MLP(self.mlp_layers)
        self.mlp_logstd = MLP(self.mlp_layers)
        
    def __call__(self, y, u):
        yu = jnp.concat((y, u), axis=-1)
        rnn_feat = self.rnn(yu).mean(axis=-2)
        return self.mlp_mean(rnn_feat), self.mlp_logstd(rnn_feat)

class Projector(nn.Module):
    outputs: int
    unflatten: Callable
    
    def setup(self):
        self.net = nn.Dense(self.outputs, use_bias=False)
        
    def __call__(self, z):
        return self.unflatten(self.net(z * 1e-1))

In [5]:
ckpt_path =  f"vae0_1.p"
# ckpt_path = Path("out") / f"hypernet2.p"
ckpt = pickle.load(open(ckpt_path, "rb"))

cfg = ckpt["cfg"]
params_enc = ckpt["params_enc"]
params_ss = ckpt["params_dec"]
params_proj = ckpt["params_proj"]
log_sigma_est = ckpt["log_sigma_est"]
sigma_est = jnp.exp(log_sigma_est)
# params_enc, params_ss, params_proj  = ckpt["params"]
scalers = ckpt["scalers"]
params_dec_flat, unflatten_dec = jax.flatten_util.ravel_pytree(params_ss)
n_params = params_dec_flat.shape[0]

In [6]:
train_lens = [100, 200, 300, 400, 500]#, 600, 700, 800, 900, 1_000, 2000, 3000, 4000, 5000]
mc_size = 100

In [7]:
data_folder = "bwdataset"
data_folder = Path(data_folder)
data = scipy.io.loadmat(data_folder / "bw_matlab.mat")

y_train = data["y"] / 7e-4
u_train = data["u"] / 50.0

y_test = scipy.io.loadmat(data_folder / "yval_multisine.mat")["yval_multisine"].reshape(-1, 1) / 7e-4
u_test = scipy.io.loadmat(data_folder / "uval_multisine.mat")["uval_multisine"].reshape(-1, 1) / 50.0
N = y_train.shape[0]

In [8]:
enc = Encoder(mlp_layers=[cfg.nh, cfg.nz], rnn_size=cfg.nh)
proj = Projector(outputs=n_params,  unflatten=unflatten_dec)

In [13]:
def loss_reduced(ov, y, u):
    
    # Project the latent into parameters of the decoder
    pe = proj.apply(params_proj, ov["z"])

    # Use in the decoder output to define/update the model parameters
    pa = jax.tree.map(lambda x, y: x+y, params_ss, pe)
    
    y_hat = ss_apply(pa, scalers, ov["x0"], u)
    #scaled_err = (y1 - y1_hat) / ckpt["sigma_noise"]
    #loss = jnp.sum(scaled_err**2) + jnp.sum(ov["z"]**2)
    loss = jnp.mean((y - y_hat)**2)/(sigma_est ** 2)
    return loss

# --- 1. Hessian Function ---
def loss_for_hessian(z, x0, y, u):
    ov = {"z": z, "x0": x0}
    return loss_reduced(ov, y, u)

batched_hessian_fn = jax.vmap(jax.hessian(loss_for_hessian, argnums=0), in_axes=(0, 0, 0, 0))

# --- 2. Predictive Uncertainty Sampling Function ---
num_post_samples = 50 # Number of samples to draw from the Laplace posterior

def get_predictive_std(z_mean, cov, key):
    # Sample z ~ N(z_mean, cov)
    z_samples = jax.random.multivariate_normal(key, mean=z_mean, cov=cov, shape=(num_post_samples,))
    
    # Map the z samples to state-space parameters
    p_ss_proj = jax.vmap(proj.apply, in_axes=(None, 0))(params_proj, z_samples)
    p_red_samples = jax.vmap(lambda p: jax.tree_util.tree_map(lambda x, y: x+y, params_ss, p))(p_ss_proj)
    
    # Simulate forward to get predictions for all samples
    x0_test = jnp.zeros((cfg.nx, ))
    y_samples = jax.vmap(ss_apply, in_axes=(0, None, None, None))(p_red_samples, scalers, x0_test, u_test)
    
    # Calculate std across the 30 samples, then average over time and output dimensions
    # Returns a single scalar representing the predictive uncertainty!
    return jnp.mean(jnp.std(y_samples, axis=0))

# Vectorize over the MC dimension
batched_get_pred_std = jax.jit(jax.vmap(get_predictive_std, in_axes=(0, 0, 0)))


def train_reduced_batched(ov_batch, y_batch, u_batch, max_iters=50_000):
    
    def single_loss_fn(ov, y, u):
        return loss_reduced(ov, y, u)
        
    # ---------------------------------------------------------
    # UPGRADED OPTIMIZATION: Cosine Decay + Gradient Clipping
    # ---------------------------------------------------------
    initial_lr = 50.0 * cfg.lr
    lr_schedule = optax.cosine_decay_schedule(
        init_value=initial_lr, 
        decay_steps=max_iters,
        alpha=1e-4  
    )
    
    opt = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=lr_schedule)
    )

    def single_make_step(ov, y, u, opt_state):
        loss, grads = jax.value_and_grad(single_loss_fn)(ov, y, u)
        updates, new_opt_state = opt.update(grads, opt_state, ov)
        new_ov = optax.apply_updates(ov, updates)
        return loss, new_ov, new_opt_state

    # Vmap over the batch dimension
    batched_make_step = jax.jit(jax.vmap(single_make_step, in_axes=(0, 0, 0, 0)))

    # --- THE FIX: Vmap the initialization! ---
    opt_state_batch = jax.vmap(opt.init)(ov_batch)
    
    losses = []
    
    # Early stopping parameters
    patience = 400 
    tol = 1e-4
    patience_counter = 0
    prev_loss = float('inf')

    # Run the Python loop with TQDM
    for idx in (pbar := tqdm(range(max_iters), desc="Adapting Batch")):
        loss_batch, ov_batch, opt_state_batch = batched_make_step(
            ov_batch, y_batch, u_batch, opt_state_batch
        )
        
        # Calculate the mean loss of the batch
        mean_loss = jnp.mean(loss_batch).item()
        losses.append(mean_loss)
        
        # Update Progress bar
        if idx % 10 == 0:
            current_lr = lr_schedule(idx) 
            pbar.set_postfix_str(f"Loss: {mean_loss:.6f} | LR: {current_lr:.2e}")

        # --- Early Stopping Logic ---
        rel_change = abs(prev_loss - mean_loss) / (prev_loss + 1e-8)
        
        if rel_change < tol:
            patience_counter += 1
        else:
            patience_counter = 0
            
        if patience_counter >= patience:
            pbar.set_postfix_str(f"Converged at step {idx}! Final Loss: {mean_loss:.6f}")
            pbar.close()
            break 
            
        prev_loss = mean_loss

    # Pad the remaining losses with NaNs so the returned array is consistent
    losses_array = onp.array(losses)
    if len(losses) < max_iters:
        padding = onp.full(max_iters - len(losses), onp.nan)
        losses_array = onp.concatenate([losses_array, padding])

    return ov_batch, jnp.array(losses_array)



In [14]:
fit_red = onp.empty((len(train_lens), mc_size))
train_time = onp.empty(len(train_lens))

pred_std_red = onp.empty((len(train_lens), mc_size)) 
latent_std_red = onp.empty((len(train_lens), mc_size))


In [16]:
# train mc_size models in parallel!

for len_idx, train_len in enumerate(train_lens):

    print(f"Processing length {train_len}...")
    
    time_start = time.time()
    # generate mc sequences
    key, subkey = jr.split(key)     
    start_indexes = jr.randint(subkey, shape=(mc_size,),  minval=0, maxval=N-train_len)
    mc_indexes = start_indexes[:, None] + jnp.arange(train_len)
    mc_y, mc_u = y_train[mc_indexes], u_train[mc_indexes]

    # print(f"Training {mc_size} reduced models...")
    # train reduced models
    # z_init = jnp.zeros((mc_size, cfg.nz, ))
    z_init,_ = jax.vmap(enc.apply, in_axes=(None, 0, 0))(params_enc, mc_y, mc_u)
    opt_vars_red_init = {"z": z_init, "x0": jnp.zeros((mc_size, cfg.nx, ))}
    
    pe = jax.vmap(proj.apply, in_axes=(None, 0))(params_proj, opt_vars_red_init["z"])

    # Use in the decoder output to define/update the model parameters
    pa = jax.tree.map(lambda x, y: x+y, params_ss, pe)
    
    x0 = jnp.zeros((cfg.nx, ))
    y_hat = jax.vmap(ss_apply, in_axes=(0,None, 0, 0))(pa, scalers, opt_vars_red_init["x0"], mc_u)
    y_test_hat = jax.vmap(ss_apply, in_axes=(0,None, None, None))(pa, scalers, x0, u_test)
    print(jnp.mean(jnp.sqrt(jnp.mean((y_hat - mc_y)**2, axis=(1, 2)))),jnp.mean(jnp.sqrt(jnp.mean((y_test_hat - y_test)**2, axis=(1, 2)))))
    # opt_vars_red, losses_red = jax.vmap(train_reduced, in_axes=(0, 0, 0))(opt_vars_red_init, mc_y, mc_u)
    opt_vars_red, losses_red = train_reduced_batched(opt_vars_red_init, mc_y, mc_u)


    # --- NEW: Plot the mean loss for this train_len ---
    import matplotlib.pyplot as plt
    
    valid_losses = losses_red[~jnp.isnan(losses_red)] # Average across the batch
    
    plt.figure(figsize=(8, 5))
    plt.plot(valid_losses, label=f"Mean Adaptation Loss (len={train_len})")
    plt.xlabel("Iterations")
    plt.ylabel("MSE Loss")
    plt.yscale("log")
    plt.title(f"Reduced Model Training Loss (Train Length: {train_len})")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"adaptation_loss_len{train_len}.png")
    plt.close()
    
    # Print the final loss using the last element of the valid array
    print(f"Final mean adaptation loss for length {train_len}: {valid_losses[-1]:.6f}")

    # project trained zetas to the ss parameter space
    params_ss_proj = jax.vmap(proj.apply, in_axes=(None, 0))(params_proj, opt_vars_red["z"])
    params_red = jax.tree.map(lambda x, y: x+y, params_ss, params_ss_proj)

    train_time[len_idx] = time.time() - time_start
    
    # NEW: Compute Laplace Approximation and Predictive Std
    # 1. Compute Covariances
    hessians = batched_hessian_fn(opt_vars_red["z"], opt_vars_red["x0"], mc_y, mc_u)*train_len
    jitter = 1e-4 * jnp.eye(cfg.nz)
    hessians_stable = hessians + jitter[None, :, :]
    covariances = jax.vmap(jnp.linalg.inv)(hessians_stable)
    
    # 2. (Optional) Store the average latent std
    variances = jax.vmap(jnp.diag)(covariances)
    stds = jnp.sqrt(jnp.abs(variances))
    latent_std_red[len_idx, :] = onp.array(jnp.mean(stds, axis=-1)) # Shape: (mc_size,)
    
    # 3. Compute the Predictive Std using the sampling function
    key, subkey = jr.split(key)
    keys = jax.random.split(subkey, mc_size)
    mc_pred_stds = batched_get_pred_std(opt_vars_red["z"], covariances, keys)
    print(mc_pred_stds.mean())
    # Store the scalar predictive std for each MC iteration!
    pred_std_red[len_idx, :] = onp.array(mc_pred_stds) # Shape: (mc_size,)


    # test reduced models
    x0 = jnp.zeros((cfg.nx, ))
    y_hat = jax.vmap(ss_apply, in_axes=(0, None, 0, 0))(params_red, scalers, opt_vars_red["x0"], mc_u)
    y_test_hat = jax.vmap(ss_apply, in_axes=(0, None, None, None))(params_red, scalers, x0, u_test)
    print(jnp.mean(jnp.sqrt(jnp.mean((y_hat - mc_y)**2, axis=(1, 2)))),jnp.mean(jnp.sqrt(jnp.mean((y_test_hat - y_test)**2, axis=(1, 2)))))

    for mc_idx in range(mc_size):
        fit_red[len_idx, mc_idx] = nonlinear_benchmarks.error_metrics.fit_index(y_test[cfg.skip_loss:], y_test_hat[mc_idx, cfg.skip_loss:])[0]

Processing length 100...


1.144147 0.5685124


Adapting Batch:  81%|████████  | 8111/10000 [01:16<00:17, 106.16it/s, Converged at step 8113! Final Loss: 0.140775]


Final mean adaptation loss for length 100: 0.140775


2026-03-17 15:14:56.812197: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


nan
0.09050289 0.48707253
Processing length 200...
0.87185544 0.50015813


Adapting Batch:  29%|██▉       | 2907/10000 [00:49<01:55, 61.54it/s, Loss: 0.360605 | LR: 4.03e-03]

In [ ]:
# Save the final checkpoint
ckpt = {
    "train_lens": train_lens,
    "train_time": train_time,
    "fit": fit_red,
    "pred_std": pred_std_red,     # <--- Add this
    "latent_std": latent_std_red  # <--- Add this
}


ckpt_path = Path("out") / f"mc_red_VAE_0_05.p"
ckpt_path.parent.mkdir(exist_ok=True, parents=True)
pickle.dump(ckpt, open(ckpt_path, "wb" ))

NameError: name 'train_lens' is not defined